#  Arabic Speech Recognition System
### Deep Learning Based Arabic Audio Understanding — Qwen3-ASR-1.7B
### Kaggle Version — Mozilla Common Voice Arabic TTS Dataset

**Pipeline:** Audio (.wav) → Qwen3-ASR-1.7B → Arabic Text → WER Evaluation → Keyword Spotting → Gradio Demo

**Dataset:** Arabic TTS (Mozilla Common Voice) — `/kaggle/input/arabic_tts/`  
**Model:** Qwen3-ASR-1.7B  
**Metrics:** Word Error Rate (WER), Character Error Rate (CER), Accuracy

---
##  Step 1 — Install Dependencies

In [24]:
import subprocess

!pip install qwen-asr librosa soundfile jiwer gradio PyWavelets

print(' All packages installed!')

 All packages installed!


---
## Step 2 — Load Dataset from Kaggle Input

In [25]:
import os
import pandas as pd
import re
import numpy as np

# ── Dataset paths ──────────────────────────────────────────────────────────
DATASET_ROOT = '/kaggle/input/datasets/mayarjao/arabic-tts'
WAV_DIR      = os.path.join(DATASET_ROOT, 'arabic_tts', 'wavs')
METADATA     = os.path.join(DATASET_ROOT, 'arabic_tts', 'metadata.csv')
METADATA_WAV = os.path.join(DATASET_ROOT, 'arabic_tts', 'metadata-wav.csv')

# Verify paths
print(' Checking paths...')
for label, path in [('Dataset root', DATASET_ROOT),
                    ('WAV directory', WAV_DIR),
                    ('metadata.csv',  METADATA),
                    ('metadata-wav.csv', METADATA_WAV)]:
    status = '' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'  {status}  {label}: {path}')

 Checking paths...
    Dataset root: /kaggle/input/datasets/mayarjao/arabic-tts
    WAV directory: /kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts/wavs
    metadata.csv: /kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts/metadata.csv
    metadata-wav.csv: /kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts/metadata-wav.csv


In [26]:
# ── Load metadata — columns are swapped in this dataset ───────────────────
print(' Loading transcript file...')

try:
    df = pd.read_csv(METADATA_WAV, sep='|', header=None,
                     names=['transcript', 'filename', 'normalized'])
    print(f' Loaded metadata-wav.csv — {len(df)} rows')
except Exception:
    df = pd.read_csv(METADATA, sep='|', header=None,
                     names=['transcript', 'filename', 'normalized'])
    print(f' Loaded metadata.csv — {len(df)} rows')

# Clean up filename — remove 'wavs/' prefix if present
df['filename'] = df['filename'].str.strip().str.replace('wavs/', '', regex=False)

# Add .wav if missing
df['filename'] = df['filename'].apply(
    lambda x: x if str(x).endswith('.wav') else str(x) + '.wav'
)

print(f'\n First 3 rows after fix:')
print(df[['filename', 'transcript']].head(3).to_string())

# Verify one file exists
test_path = os.path.join(WAV_DIR, df.iloc[0]['filename'])
print(f' Test path exists: {os.path.exists(test_path)}')
print(f'   {test_path}')

 Loading transcript file...
 Loaded metadata-wav.csv — 78720 rows

 First 3 rows after fix:
                       filename                    transcript
0  common_voice_ar_24203362.wav    هو يحبّها و هي تحبّه أيضا.
1  common_voice_ar_22931432.wav  من الممكن أنها لن تأتي غداً.
2  common_voice_ar_26338992.wav                     إبنك بطل.
 Test path exists: True
   /kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts/wavs/common_voice_ar_24203362.wav


In [27]:
# ── Normalization ─────────────────────────────────────────────────────────
def normalize(text):
    """Normalize Arabic text for fair WER comparison."""
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)  # strip diacritics
    text = re.sub(r'[أإآٱ]', 'ا', text)                   # unify alef
    text = text.replace('ة', 'ه')                          # teh marbuta
    text = text.replace('ى', 'ي')                          # alef maqsura
    text = re.sub(r'[.،؟!,?"\']', '', text)              # punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_prediction(text):
    """Same normalization applied to model output."""
    if isinstance(text, bytes):
        text = text.decode('utf-8')
    if not isinstance(text, str):
        text = str(text)
    return normalize(text)

# ── Build samples list ─────────────────────────────────────────────────────
print(' Building dataset...')
samples = []

for _, row in df.iterrows():
    filename = str(row['filename']).strip()

    # Add .wav if not present
    if not filename.endswith('.wav'):
        filename = filename + '.wav'

    audio_path = os.path.join(WAV_DIR, filename)

    # Use normalized transcript if available, else raw
    if pd.notna(row.get('normalized', None)) and str(row['normalized']).strip():
        text = normalize(str(row['normalized']))
    else:
        text = normalize(str(row['transcript']))

    if os.path.exists(audio_path) and text:
        samples.append({
            'file_id':        filename,
            'audio_path':     audio_path,
            'reference_text': text
        })

# 70% train / 30% test — same as paper
import random
random.seed(42)
random.shuffle(samples)

split_idx     = int(len(samples) * 0.7)
train_samples = samples[:split_idx]
test_samples  = samples[split_idx:]

print(f' Dataset Summary:')
print(f'   Total samples  : {len(samples)}')
print(f'   Training (70%) : {len(train_samples)}')
print(f'   Testing  (30%) : {len(test_samples)}')
print(f' Example:')
print(f'   File : {test_samples[0]["file_id"]}')
print(f'   Text : {test_samples[0]["reference_text"]}')
print(f'   Path : {test_samples[0]["audio_path"]}')

 Building dataset...
 Dataset Summary:
   Total samples  : 78720
   Training (70%) : 55104
   Testing  (30%) : 23616
 Example:
   File : common_voice_ar_19233876.wav
   Text : لدي ثلاث اولاد خاله
   Path : /kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts/wavs/common_voice_ar_19233876.wav


---
##  Step 3 — Audio Exploration

In [28]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from IPython.display import Audio, display

sample = test_samples[0]
y, sr  = librosa.load(sample['audio_path'], sr=None, mono=True)
dur    = len(y) / sr

print(f' Reference : {sample["reference_text"]}')
print(f' SR        : {sr} Hz')
print(f'  Duration  : {dur:.2f} seconds')

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')
fig.suptitle(f'Audio Analysis\n{sample["reference_text"]}',
             color='white', fontsize=11)
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('#30363d')

librosa.display.waveshow(y, sr=sr, ax=axes[0], color='#58a6ff')
axes[0].set_title('Waveform', color='white')
axes[0].set_xlabel('Time (s)', color='#8b949e')
axes[0].set_ylabel('Amplitude', color='#8b949e')

mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=128)
img  = librosa.display.specshow(mfcc, sr=sr, x_axis='time',
                                 ax=axes[1], cmap='magma')
axes[1].set_title('MFCC Features (128 coefficients — same as paper)',
                   color='white')
axes[1].set_xlabel('Time (s)', color='#8b949e')
axes[1].set_ylabel('MFCC Coefficients', color='#8b949e')
plt.colorbar(img, ax=axes[1])
plt.tight_layout()
plt.savefig('/kaggle/working/audio_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
display(Audio(sample['audio_path']))
print(' Saved → /kaggle/working/audio_analysis.png')

 Reference : لدي ثلاث اولاد خاله
 SR        : 22050 Hz
  Duration  : 3.46 seconds


 Saved → /kaggle/working/audio_analysis.png


---
##  Step 4 — Load Qwen3-ASR-1.7B Model

In [29]:
from qwen_asr import Qwen3ASRModel
import torch

print(' Loading Qwen3-ASR-1.7B...')
print('   First run downloads ~4.7GB — takes 5-10 minutes...\n')

model = Qwen3ASRModel.from_pretrained(
    'Qwen/Qwen3-ASR-1.7B',
    dtype=torch.bfloat16,
    device_map='cuda:0' if torch.cuda.is_available() else 'cpu',
    max_inference_batch_size=8,
    max_new_tokens=256,
)

print(' Qwen3-ASR-1.7B loaded successfully!')

 Loading Qwen3-ASR-1.7B...
   First run downloads ~4.7GB — takes 5-10 minutes...



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 Qwen3-ASR-1.7B loaded successfully!


---
##  Step 5 — Inference & WER Evaluation

In [30]:
import soundfile as sf
import tempfile
import librosa
import numpy as np
from jiwer import wer, cer

TARGET_SR = 16000

def transcribe_audio(audio_path):
    """
    Load wav file, resample to 16kHz, transcribe with Qwen3-ASR.
    Returns normalized Arabic text.
    """
    # Load and resample
    y, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    y = np.array(y, dtype=np.float32)

    # Save to temp file — Qwen3-ASR needs a file path
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        sf.write(tmp.name, y, TARGET_SR)
        tmp_path = tmp.name

    # Transcribe
    results = model.transcribe(tmp_path, language='Arabic')
    text    = results[0].text.strip()

    # Clean up temp file
    try:
        os.remove(tmp_path)
    except Exception:
        pass

    return text


# ── Quick single-sample test ───────────────────────────────────────────────
print('🧪 Quick test on first sample...')
s    = test_samples[0]
pred = normalize_prediction(transcribe_audio(s['audio_path']))
ref  = s['reference_text']

print(f'\n   Reference  : {ref}')
print(f'   Prediction : {pred}')
print(f'   Sample WER : {wer(ref, pred)*100:.1f}%')

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


🧪 Quick test on first sample...

   Reference  : لدي ثلاث اولاد خاله
   Prediction : لدي ثلاثه اولاد خاله
   Sample WER : 25.0%


In [38]:
from tqdm import tqdm
import pandas as pd

#  Set MAX_EVAL to None to evaluate ALL test samples
MAX_EVAL = 100

eval_samples                                    = test_samples[:MAX_EVAL] if MAX_EVAL else test_samples
references, hypotheses, results_log, errors     = [], [], [], []

print(f' Evaluating {len(eval_samples)} test samples...\n')

for sample in tqdm(eval_samples, desc='Transcribing'):
    try:
        pred = normalize_prediction(transcribe_audio(sample['audio_path']))
        ref  = sample['reference_text']

        references.append(ref)
        hypotheses.append(pred)
        results_log.append({
            'file_id':    sample['file_id'],
            'reference':  ref,
            'prediction': pred,
            'wer':        round(wer(ref, pred), 4)
        })
    except Exception as e:
        errors.append({'file_id': sample['file_id'], 'error': str(e)})

overall_wer = wer(references, hypotheses)
overall_cer = cer(references, hypotheses)
accuracy    = 1 - overall_wer

print(f'\n{"="*52}')
print(f'    EVALUATION RESULTS')
print(f'{"="*52}')
print(f'  Samples evaluated : {len(references)}')
print(f'  Word Error Rate   : {overall_wer*100:.2f}%')
print(f'  Char Error Rate   : {overall_cer*100:.2f}%')
print(f'  Accuracy (1-WER)  : {accuracy*100:.2f}%')
print(f'  Failed samples    : {len(errors)}')
print(f'{"="*52}')
print(f'  Qwen3-ASR-1.7B  — Accuracy: {accuracy*100:.2f}%  | WER: {overall_wer*100:.2f}%')

 Evaluating 100 test samples...



Transcribing: 100%|██████████| 100/100 [01:30<00:00,  1.11it/s]


    EVALUATION RESULTS
  Samples evaluated : 100
  Word Error Rate   : 18.39%
  Char Error Rate   : 5.92%
  Accuracy (1-WER)  : 81.61%
  Failed samples    : 0
  Qwen3-ASR-1.7B  — Accuracy: 81.61%  | WER: 18.39%


## Manual verification  for WER to check if it real value or not 


In [32]:
# ── Manual verification — check 10 samples visually ───────────────────────
print('Manual Verification — 10 random samples\n')
print(f'{"─"*70}')

import random
random.seed(123)
sample_check = random.sample(results_log, 10)

for i, row in enumerate(sample_check):
    print(f'\nSample {i+1}: {row["file_id"]}')
    print(f'   Reference  : {row["reference"]}')
    print(f'   Prediction : {row["prediction"]}')
    print(f'   WER        : {row["wer"]*100:.1f}%')
    print(f'  {" Good" if row["wer"] < 0.3 else "  Difficult" if row["wer"] < 0.7 else "❌ Poor"}')

print(f'\n{"─"*70}')
print(f'\n Distribution check:')
perfect   = sum(1 for r in results_log if r["wer"] == 0.0)
good      = sum(1 for r in results_log if 0.0 < r["wer"] <= 0.3)
moderate  = sum(1 for r in results_log if 0.3 < r["wer"] <= 0.7)
poor      = sum(1 for r in results_log if r["wer"] > 0.7)

print(f'  Perfect (WER=0%)    : {perfect} samples')
print(f'  Good    (WER≤30%)   : {good} samples')
print(f'  Moderate(WER≤70%)   : {moderate} samples')
print(f'  Poor    (WER>70%)   : {poor} samples')

# Double check overall WER manually
from jiwer import wer
manual_refs  = [r['reference']  for r in results_log]
manual_preds = [r['prediction'] for r in results_log]
manual_wer   = wer(manual_refs, manual_preds)
print(f'\n Manual WER recomputed : {manual_wer*100:.2f}%')
print(f'   Matches reported WER  : {abs(manual_wer - overall_wer) < 0.001}')

Manual Verification — 10 random samples

──────────────────────────────────────────────────────────────────────

Sample 1: common_voice_ar_24067845.wav
   Reference  : يا الله اترك لي بصيص امل اعيش به باقي ايامي فانني قد وهنت
   Prediction : يا الله اترك لي بصيص امل اعيش به باقي ايامي فانني قد وهنت
   WER        : 0.0%
   Good

Sample 2: common_voice_ar_24029080.wav
   Reference  : قال سلام عليك ساستغفر لك ربي انه كان بي حفيا
   Prediction : قال سلام عليك ساستغفر لك رب انه كان بيحفيا
   WER        : 30.0%
    Difficult

Sample 3: common_voice_ar_24147288.wav
   Reference  : ارسل لك سامي رساله
   Prediction : ارسل لك سامي رساله
   WER        : 0.0%
   Good

Sample 4: common_voice_ar_21629218.wav
   Reference  : انها تقوم بصرف الكثير من المال علي الكتب
   Prediction : انها تقوم بصرف الكثير من المال علي الكتب
   WER        : 0.0%
   Good

Sample 5: common_voice_ar_24030710.wav
   Reference  : الطقس اليوم شديد الحراره
   Prediction : طقس اليوم شديد الحراره
   WER        : 25.0%
   Good

Sa

In [33]:
results_df = pd.DataFrame(results_log).sort_values('wer')

print('Best predictions (lowest WER):')
display(results_df.head(10).reset_index(drop=True))
print('Worst predictions (highest WER):')
display(results_df.tail(10).reset_index(drop=True))

results_df.to_csv('/kaggle/working/asr_evaluation_results.csv',
                  index=False, encoding='utf-8')
print(' Saved → /kaggle/working/asr_evaluation_results.csv')

Best predictions (lowest WER):


,file_id,reference,prediction,wer
0,common_voice_ar_24057632.wav,وقال الجاحظ,وقال الجاحظ,0.0
1,common_voice_ar_24086446.wav,ويكون بحسب اختلاف الرغبه فيما خيف عليه,ويكون بحسب اختلاف الرغبه فيما خيف عليه,0.0
2,common_voice_ar_24067845.wav,يا الله اترك لي بصيص امل اعيش به باقي ايامي فا...,يا الله اترك لي بصيص امل اعيش به باقي ايامي فا...,0.0
3,common_voice_ar_22404712.wav,اربعه,اربعه,0.0
4,common_voice_ar_19360294.wav,التلفاز مفتوح,التلفاز مفتوح,0.0
5,common_voice_ar_24147288.wav,ارسل لك سامي رساله,ارسل لك سامي رساله,0.0
6,common_voice_ar_24075959.wav,تدل اسئلتها علي اطلاعها الجيد علي الموضوع,تدل اسئلتها علي اطلاعها الجيد علي الموضوع,0.0
7,common_voice_ar_24046835.wav,ماذا افعل بالسكين,ماذا افعل بالسكين,0.0
8,common_voice_ar_24165205.wav,هل انت جديد هنا,هل انت جديد هنا,0.0
9,common_voice_ar_24035547.wav,اخيرا وجدت وظيفه,اخيرا وجدت وظيفه,0.0


Worst predictions (highest WER):


,file_id,reference,prediction,wer
0,common_voice_ar_24055299.wav,اريد ان اقضي وقتا اكثر معك وحدنا,واذ نقلي وقتا اكثر ماكوحتنه,0.7143
1,common_voice_ar_24159556.wav,احقا سيذهب توم الي بوسطن لوحده,احقا سيدك بصام لبصني واحده,0.8333
2,common_voice_ar_19238849.wav,لجون ابنان,ليجون لبنان,1.0000
3,common_voice_ar_24052358.wav,وما هو بالهزل,وله بالعزل,1.0000
4,common_voice_ar_24160268.wav,نظف ثيابك,نظفت يابك,1.0000
5,common_voice_ar_24041178.wav,اريدك,اوريدو,1.0000
6,common_voice_ar_24167580.wav,وارسلناه الي مائه الف او يزيدون,وارسله لماتالف ويزيدون,1.0000
7,common_voice_ar_24148349.wav,اذ انبعث اشقاها,اذا بعت اشقها,1.0000
8,common_voice_ar_19532657.wav,احذر,احداب,1.0000
9,common_voice_ar_24074076.wav,هدي ورحمه للمحسنين,وهدوا رحمه من المحسنين,1.3333


 Saved → /kaggle/working/asr_evaluation_results.csv


In [34]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('ASR Evaluation Results', color='white', fontsize=14, fontweight='bold')
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_color('#30363d')

# WER distribution
axes[0].hist(results_df['wer'].values, bins=20,
             color='#58a6ff', edgecolor='#0d1117', alpha=0.85)
axes[0].axvline(overall_wer, color='#f85149', lw=2, ls='--',
                label=f'Mean WER={overall_wer*100:.1f}%')
axes[0].axvline(0.0469, color='#3fb950', lw=2, ls='--',
                label='Paper WER=4.69%')
axes[0].set_title('WER Distribution', color='white')
axes[0].set_xlabel('Word Error Rate', color='#8b949e')
axes[0].set_ylabel('Number of Samples', color='#8b949e')
axes[0].legend(facecolor='#161b22', labelcolor='white')

# Model comparison
models   = ['MLP\n[Paper]','LSTM\n[Paper]','CNN-LSTM\n[Paper]',
            'GRU\n[Paper]','Qwen3-ASR\n(Ours)']
acc_vals = [65.72, 71.58, 93.0, 95.31, accuracy*100]
colors   = ['#6e7681','#6e7681','#6e7681','#3fb950','#58a6ff']
bars     = axes[1].bar(models, acc_vals, color=colors,
                        edgecolor='#0d1117', width=0.6)
axes[1].set_title('Model Accuracy Comparison', color='white')
axes[1].set_ylabel('Accuracy (%)', color='#8b949e')
axes[1].set_ylim(0, 110)
for bar, val in zip(bars, acc_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f}%', ha='center', va='bottom',
                 color='white', fontsize=9)

plt.tight_layout()
plt.savefig('/kaggle/working/evaluation_results.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('💾 Saved → /kaggle/working/evaluation_results.png')

💾 Saved → /kaggle/working/evaluation_results.png


---
##  Step 7 — Keyword Spotting

In [35]:
from collections import defaultdict

KEYWORDS = {
    'مرحبا':   'greeting',   'السلام':  'greeting',
    'صباح':    'greeting',   'مساء':    'greeting',
    'اهلا':    'greeting',
    'واحد':    'number',     'اثنان':   'number',
    'ثلاثه':   'number',     'اربعه':   'number',
    'خمسه':    'number',     'سته':     'number',
    'سبعه':    'number',     'ثمانيه':  'number',
    'تسعه':    'number',     'عشره':    'number',
    'يمين':    'direction',  'يسار':    'direction',
    'كيف':     'question',   'اين':     'question',
    'ماذا':    'question',   'متي':     'question',
    'لا':      'negation',   'نعم':     'affirmation',
    'مريض':    'health',
    'مساعده':  'emergency',  'طوارئ':   'emergency',
}

def spot_keywords(text, keywords=KEYWORDS):
    text_norm = normalize(text)
    return [{'keyword': kw, 'category': cat}
            for kw, cat in keywords.items() if kw in text_norm]


# Run on all evaluated samples
kw_results      = []
category_counts = defaultdict(int)

for row in results_log:
    found = spot_keywords(row['prediction'])
    kw_results.append({
        'file_id':        row['file_id'],
        'transcript':     row['prediction'],
        'keywords_found': [k['keyword']  for k in found],
        'categories':     [k['category'] for k in found],
        'count':          len(found)
    })
    for k in found:
        category_counts[k['category']] += 1

kw_df      = pd.DataFrame(kw_results)
kw_df_hits = kw_df[kw_df['count'] > 0]

print(f'  Samples with keywords : {len(kw_df_hits)} / {len(kw_df)}')
print(f'  Total keyword hits    : {int(kw_df["count"].sum())}')
print(f'\n  By category:')
for cat, cnt in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f'    {cat:15s}: {cnt}')

if len(kw_df_hits) > 0:
    display(kw_df_hits[['file_id','transcript','keywords_found']].head(10))

kw_df.to_csv('/kaggle/working/keyword_spotting_results.csv',
             index=False, encoding='utf-8')
print(' Saved → /kaggle/working/keyword_spotting_results.csv')

  Samples with keywords : 32 / 100
  Total keyword hits    : 34

  By category:
    negation       : 25
    number         : 5
    question       : 3
    affirmation    : 1


,file_id,transcript,keywords_found
0,common_voice_ar_19233876.wav,لدي ثلاثه اولاد خاله,"[ثلاثه, لا]"
2,common_voice_ar_24086446.wav,ويكون بحسب اختلاف الرغبه فيما خيف عليه,[لا]
3,common_voice_ar_24146124.wav,لا ينبغي لذ الوجهين ان يكون وجهان عند الله تعالي,[لا]
4,common_voice_ar_22404712.wav,اربعه,[اربعه]
5,common_voice_ar_24439505.wav,ثمانيه ادار يوم المراه اليوم هو يومنا مبروك لنا,[ثمانيه]
7,common_voice_ar_24042203.wav,نحن لا نتوعد,[لا]
9,common_voice_ar_24046835.wav,ماذا افعل بالسكين,[ماذا]
10,common_voice_ar_24075959.wav,تدل اسئلتها علي اطلاعها الجيد علي الموضوع,[لا]
17,common_voice_ar_24159556.wav,احقا سيدك بصام لبصني واحده,[واحد]
18,common_voice_ar_24157700.wav,لماذا تبكين يا ماري,[ماذا]


 Saved → /kaggle/working/keyword_spotting_results.csv


---
##  Step 8 — Gradio Demo Interface

In [36]:
import gradio as gr

def run_pipeline(audio_input):
    if audio_input is None:
        return ' Please upload an audio file.', '', ''
    try:
        transcript = normalize_prediction(transcribe_audio(audio_input))
        found      = spot_keywords(transcript)
        kw_text    = '\n'.join(
            [f'  {k["keyword"]}   [{k["category"]}]' for k in found]
        ) if found else '  No keywords detected'

        y, sr = librosa.load(audio_input, sr=None)
        dur   = len(y) / sr
        info  = (f'Duration: {dur:.2f}s  |  SR: {sr} Hz  |  '
                 f'Model: Qwen3-ASR-1.7B')
        return transcript, kw_text, info
    except Exception as e:
        return f' Error: {str(e)}', '', ''


with gr.Blocks(
    title='Arabic ASR — Qwen3-ASR-1.7B',
    theme=gr.themes.Base(
        primary_hue='blue',
        neutral_hue='slate',
        font=[gr.themes.GoogleFont('IBM Plex Sans'), 'sans-serif']
    )
) as demo:

    gr.HTML("""
    <div style='text-align:center; padding:24px 0 12px'>
      <h1 style='color:#38bdf8; margin:0; font-size:2em'>
        🎙️ Arabic ASR System
      </h1>
      <p style='color:#94a3b8; margin:10px 0 0'>
        Powered by Qwen3-ASR-1.7B &nbsp;·&nbsp;
        Dataset: Mozilla Common Voice Arabic TTS &nbsp;·&nbsp;
        Speech → Arabic Text + Keyword Spotting
      </p>
    </div>""")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### 🔊 Input')
            audio_in = gr.Audio(
                label='Upload Arabic Audio (.wav) or Record',
                type='filepath',
                sources=['upload', 'microphone']
            )
            run_btn = gr.Button('▶  Transcribe', variant='primary', size='lg')

        with gr.Column(scale=1):
            gr.Markdown('###  Output')
            transcript_out = gr.Textbox(
                label='Arabic Transcription',
                lines=4, rtl=True,
                placeholder='النص العربي سيظهر هنا...'
            )
            keywords_out = gr.Textbox(
                label='Detected Keywords',
                lines=4,
                placeholder='Keywords will appear here...'
            )
            info_out = gr.Textbox(
                label='Audio Info',
                lines=1,
                interactive=False
            )

    run_btn.click(
        fn=run_pipeline,
        inputs=[audio_in],
        outputs=[transcript_out, keywords_out, info_out]
    )

    gr.Markdown(f"""
---
| Property | Value |
|---|---|
| Model | Qwen3-ASR-1.7B |
| Language | Arabic |
| Dataset | Mozilla Common Voice Arabic TTS |
| Test Samples | {len(test_samples)} |
| Evaluated | {len(references)} |
| Word Error Rate | {overall_wer*100:.2f}% |
| Accuracy | {accuracy*100:.2f}% |
""")

demo.launch(share=True, debug=False)
print(' Demo is live! Use the public link above.')

/tmp/ipykernel_57/1547212753.py:22: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://698c71dcebd5de4d47.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


 Demo is live! Use the public link above.


---
##  Step 9 — Final Summary Report

In [37]:
print('=' * 58)
print('        FINAL PROJECT SUMMARY REPORT')
print('=' * 58)
print(f"""
Model      : Qwen3-ASR-1.7B
Dataset    : Mozilla Common Voice Arabic TTS
Location   : /kaggle/input/arabic_tts/

DATASET
  Total samples  : {len(samples)}
  Training (70%) : {len(train_samples)}
  Testing  (30%) : {len(test_samples)}
  Audio format   : WAV

EVALUATION (Test Set)
  Samples evaluated : {len(references)}
  Word Error Rate   : {overall_wer*100:.2f}%
  Char Error Rate   : {overall_cer*100:.2f}%
  Accuracy (1-WER)  : {accuracy*100:.2f}%
  Failed samples    : {len(errors)}

COMPARISON WITH PAPER (Alahmadi et al., 2024)
  MLP autoencoder        : 65.72% accuracy
  LSTM autoencoder       : 71.58% accuracy  | WER 28.42%
  CNN-LSTM               : 93.00% accuracy  | WER 13.52%
  GRU autoencoder [BEST] : 95.31% accuracy  | WER  4.69%
  Qwen3-ASR-1.7B  (Ours) : {accuracy*100:.2f}% accuracy  | WER  {overall_wer*100:.2f}%

KEYWORD SPOTTING
  Keywords defined   : {len(KEYWORDS)}
  Samples with hits  : {len(kw_df_hits)} / {len(kw_df)}
  Total keyword hits : {int(kw_df['count'].sum())}

OUTPUT FILES
  asr_evaluation_results.csv    → /kaggle/working/
  keyword_spotting_results.csv  → /kaggle/working/
  evaluation_results.png        → /kaggle/working/
  audio_analysis.png            → /kaggle/working/

DELIVERABLES
   Source code        : this notebook
   Dataset loading    : Mozilla Common Voice Arabic TTS
   Model inference    : Qwen3-ASR-1.7B
   Evaluation results : WER + Accuracy
   Keyword spotting   : Step 7
   Demo interface     : Gradio (Step 8)
""")
print('=' * 58)

        FINAL PROJECT SUMMARY REPORT

Model      : Qwen3-ASR-1.7B
Dataset    : Mozilla Common Voice Arabic TTS
Location   : /kaggle/input/arabic_tts/

DATASET
  Total samples  : 78720
  Training (70%) : 55104
  Testing  (30%) : 23616
  Audio format   : WAV

EVALUATION (Test Set)
  Samples evaluated : 100
  Word Error Rate   : 18.39%
  Char Error Rate   : 5.92%
  Accuracy (1-WER)  : 81.61%
  Failed samples    : 0

COMPARISON WITH PAPER (Alahmadi et al., 2024)
  MLP autoencoder        : 65.72% accuracy
  LSTM autoencoder       : 71.58% accuracy  | WER 28.42%
  CNN-LSTM               : 93.00% accuracy  | WER 13.52%
  GRU autoencoder [BEST] : 95.31% accuracy  | WER  4.69%
  Qwen3-ASR-1.7B  (Ours) : 81.61% accuracy  | WER  18.39%

KEYWORD SPOTTING
  Keywords defined   : 26
  Samples with hits  : 32 / 100
  Total keyword hits : 34

OUTPUT FILES
  asr_evaluation_results.csv    → /kaggle/working/
  keyword_spotting_results.csv  → /kaggle/working/
  evaluation_results.png        → /kaggle/worki